# PT & OT Consult Orders
Creates CLIF *key_icu_orders* table from MIMIC data.


In [1]:
import pandas as pd
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

#Load data
pt_df = helper.load_data("mimic","chartevents", folder='icu')

#Filter for PT orders only
chart_mask = \
    pt_df["hadm_id"].notna() &\
    pt_df["itemid"].notna() &\
    (pt_df["itemid"] == 225135) &\
    pt_df["value"].notna() &\
    pt_df["value"].isin(["PT","OT"])
pt_df = pt_df[chart_mask]
pt_df.shape

📢 ClifOrchestrator initialized


(40996, 11)

In [3]:
#Convert string data types
pt_df['hospitalization_id'] = pt_df['hadm_id'].astype(str)
pt_df['order_status'] = 'sent'

#Convert times
pt_df['order_dttm'] = pd.to_datetime(pt_df['charttime'],utc=False)
pt_df['order_dttm'] = pt_df['order_dttm'].dt.tz_localize('America/New_York', ambiguous=True, nonexistent='shift_forward')
pt_df['order_dttm'] = pt_df['order_dttm'].dt.tz_convert('UTC')
print(f"Blank order times: {sum(pt_df['order_dttm'].isna())}")

Blank order times: 0


In [4]:
#Convert order name and categorize
pt_df['value'].value_counts()
pt_df['order_name'] = pt_df["value"]
mapping = {
    "PT": "pt_evaluation",
    "OT": "ot_evaluation"
}
pt_df['order_category'] = pt_df['order_name'].map(mapping)
pt_df['order_category'].value_counts()

order_category
pt_evaluation    29203
ot_evaluation    11793
Name: count, dtype: int64

In [5]:
#Reorganize columns
pt_df = pt_df[['hospitalization_id','order_dttm','order_name','order_category','order_status']]
pt_df.dtypes

hospitalization_id                 object
order_dttm            datetime64[ns, UTC]
order_name                         object
order_category                     object
order_status                       object
dtype: object

In [6]:
pt_df.head()

,hospitalization_id,order_dttm,order_name,order_category,order_status
16487,26184834,2131-01-11 23:37:00+00:00,OT,ot_evaluation,sent
16488,26184834,2131-01-11 23:37:00+00:00,PT,pt_evaluation,sent
16526,26184834,2131-01-11 23:54:00+00:00,OT,ot_evaluation,sent
16527,26184834,2131-01-11 23:54:00+00:00,PT,pt_evaluation,sent
39310,22725460,2112-12-05 10:23:00+00:00,OT,ot_evaluation,sent


In [8]:
#Save
import os
path = os.path.join(helper.path_out, "clif_key_icu_orders.parquet")
pt_df.to_parquet(path)